미리 정의하지 않은 애트리뷰트에 대한 접근을 지원하기 위해서, 파이썬 `__getattr__` 특별 메서드를 사용 가능하다.

In [ ]:
class LazyRecord:
    def __init__(self):
        self.exists = 5

    def __getattr__(self, name):
        value = f'{name}의 값'
        setattr(self, name, value)
        return value

data = LazyRecord()
print('이전:', data.__dict__)
print('foo:', data.foo)
print('이후:', data.__dict__)

`__getattr__`는 계속 호출되지 않는다. 이미 `foo`가 정의되어 있는 상태라서, 바로 접근이 되는걸 볼 수 있다.

In [ ]:
class LoggingLazyRecord(LazyRecord):
    def __getattr__(self, name):
        print(f'* 호출: __getattr__({name!r}), '
              f'인스턴스 딕셔너리 채워 넣음')
        result = super().__getattr__(name)
        print(f'* 반환: {result!r}')
        return result

data = LoggingLazyRecord()
print('exists: ', data.exists)
print('첫 번째 foo: ', data.foo)
print('두 번째 foo: ', data.foo)

어트리뷰트에 접근할 때마다 호출하고 싶다면 `__getattribute__` 특별 메서드를 사용하면 된다.

In [ ]:
class ValidatingRecord:
    def __init__(self):
        self.exists = 5
    def __getattribute__(self, name):
        print(f'* 호출: __getattr__({name!r})')
        try:
            value = super().__getattribute__(name)
            print(f'* {name!r} 찾음, {value!r} 반환')
            return value
        except AttributeError:
            value = f'{name}을 위한 값'
            print(f'* {name!r}를 {value!r}로 설정')
            setattr(self, name, value)
            return value

data = ValidatingRecord()
print('exists: ', data.exists)
print('첫 번째 foo: ', data.foo)
print('두 번째 foo: ', data.foo)

존재하지 않는 어트리뷰트에 접근할때는 `AttributeError`가 발생한다.

In [ ]:
class MissingPropertyRecord:
    def __getattr__(self, name):
        if name == 'bad_name':
            raise AttributeError(f'{name}을 찾을 수 없음')
        return 1 # 무조건 1 반환

data = MissingPropertyRecord()
data.bad_name

`hasattr` 내장 함수를 사용하여 프로퍼티가 존재하는지 검사할 수 있다. 여기도 `__getattr__`은 단 한 번만 호출된다.

In [ ]:
data = LoggingLazyRecord() # __getattr__을 구현
print('이전: ', data.__dict__)
print('최초에 foo가 있나: ', hasattr(data, 'foo'))
print('이후: ', data.__dict__)
print('다음에 foo가 있나: ', hasattr(data, 'foo'))

`__getattribute__`를 구현한 클래스에서는 매번 호출되는 것을 알 수 있다.

In [ ]:
data = ValidatingRecord() # __getattribute__를 구현
print('최초에 foo가 있나: ', hasattr(data, 'foo'))
print('다음에 foo가 있나: ', hasattr(data, 'foo'))

임의의 애트리뷰트에 값을 설정할 때마다 `__setattr__`이 호출된다.

In [ ]:
class SavingRecord:
    def __setattr__(self, name, value):
        # 데이터를 데이터베이스 레코드에 저장한다
        super().__setattr__(name, value)

class LoggingSavingRecord(SavingRecord):
    def __setattr__(self, name, value):
        print(f'* 호출: __setattr__({name!r}, {value!r})')
        super().__setattr__(name, value)

data = LoggingSavingRecord()
print('이전: ', data.__dict__)
data.foo = 5
print('이후: ', data.__dict__)
data.foo = 7
print('최후:', data.__dict__)

이렇게 매번 수행되는 걸로 인해 원하지 않던 오류가 발생할 수 있다. `__getattribute__`가 `self._data`에 접근해서 `__getattribute__`가 호출되면서 재귀함수와 같이 동작하고 _stack overflow_ 가 발생한다.

In [ ]:
class BrokenDictionaryRecord:
    def __init__(self, data):
        self._data = {}
        
    def __getattribute__(self, name):
        print(f'* 호출: __getattribute__({name!r})')
        return self._data[name]

data = Brokedata = BrokenDictionaryRecord({'foo': 3})
data.foo

`super().__getattribute__`를 호출해 인스턴스 애트리뷰트 딕셔너리에서 값을 가져오면 해결된다.

In [ ]:
class DictionaryRecord:
    def __init__(self, data):
        self._data = data

    def __getattribute__(self, name):
        print(f'* 호출: __getattribute__({name!r})')
        data_dict = super().__getattribute__('_data')
        return data_dict[name]

data = DictionaryRecord({'foo': 3})
print('foo: ', data.foo)